In [ ]:
# to fix imports 
import sys; from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

# 1. Revision Pinning

The dataset dump updates frequently so we need to pin the version that we currently have in time  
to achieve this, we need to get the current revision hash from hugging face

In [ ]:
from datetime import datetime, timezone

from huggingface_hub import HfApi
from src.config import HF_REPO_ID

api = HfApi()
info = api.dataset_info(HF_REPO_ID)

print(f"Repo: {HF_REPO_ID}")
print(f"Revision SHA: {info.sha}")
print(f"Last modified: {info.last_modified}")
print(f"Current time (UTC): {datetime.now(timezone.utc).isoformat()}")


In [ ]:
# the original output:

# Repo: openfoodfacts/product-database
# Revision SHA: e49264ae39d1fb35f7abbc4915dbe715af06345b
# Last modified: 2026-11-23 11:30:29+00:00
# Current time (UTC): 2026-11-26T06:06:13.924530+00:00

copy the revision hash and place it on config.py

# 2. Raw Dataset Download

initial solution:
use duckdb http stream to avoid downloading 7gb and directly copying relevant rows and columns only  

problems:  
kernel dies mid analysis/copy (mostly out of ram/connection issues)  
very slow (20 minutes of waiting just to error)  

solution: download the whole dataset instead and run duckdb locally  
github codespace has enough storage and we will delete the raw data later  

In [ ]:
from huggingface_hub import HfFileSystem

from src.config import (
    HF_REMOTE_URL,
    HF_TOKEN,
    RAW_DIR,
    RAW_PARQUET_PATH,
)

RAW_DIR.mkdir(parents=True, exist_ok=True)

if RAW_PARQUET_PATH.exists():
    print(f"Raw dataset already exists, skipping download: {RAW_PARQUET_PATH}")
else:
    filesystem = HfFileSystem(token=HF_TOKEN)
    filesystem.get_file(HF_REMOTE_URL, str(RAW_PARQUET_PATH))

# 3. Raw Data Analysis

We will use duckdb to speed up the process since the raw data is huge

In [ ]:
# initialize duckdb connection and set options
import duckdb
import pandas as pd

from src.config import DUCKDB_TEMP_DIR

pd.set_option("display.max_rows", None)

DUCKDB_TEMP_DIR.mkdir(parents=True, exist_ok=True)

connection = duckdb.connect()
connection.execute("SET enable_progress_bar = true")
connection.execute("SET memory_limit = '2GB'")
connection.execute("SET threads = 2")
connection.execute("SET preserve_insertion_order = false")
connection.execute(f"SET temp_directory = '{DUCKDB_TEMP_DIR.as_posix()}'")

raw_source = f"read_parquet('{RAW_PARQUET_PATH.as_posix()}')"

## 3.1 Row Examples

get samples from dataset to have an idea how it looks like.
alternatively, go to huggingface data studio to view rows

In [ ]:
sample_limit = 5
sample = connection.sql(f"SELECT * FROM {raw_source} LIMIT {sample_limit}")
sample.df()

## 3.2 Row Description

check total rows and columns of raw dataset

In [ ]:
total_rows = connection.execute(f"SELECT COUNT(*) FROM {raw_source}").fetchone()[0]
schema_rows = connection.execute(f"DESCRIBE SELECT * FROM {raw_source}").fetchall()
total_columns = len(schema_rows)

print(f"rows: {total_rows:,}")
print(f"columns: {total_columns}")

findings:
4 million rows x 111 columns is large for our hardware 
need to get the ph based products only, not worldwide  
so we will create a smaller dataset from it instead  

## 3.3 Column Analysis

display all the columns and their datatypes

In [ ]:
schema = connection.sql(f"DESCRIBE SELECT * FROM {raw_source}")
schema.df()

findings:
keep for now, cleaning process will be on the next notebook

# 4. Extract Philippines Based Product Subset

Keep only the rows tagged `en:philippines` in `countries_tags` and save them as the PH raw parquet

In [ ]:
from src.config import INTERIM_DIR, PH_RAW_PARQUET_PATH

INTERIM_DIR.mkdir(parents=True, exist_ok=True)

if PH_RAW_PARQUET_PATH.exists():
    print(f"PH subset already exists, skipping: {PH_RAW_PARQUET_PATH}")
else:
    connection.execute(
        f"""
        COPY (
            SELECT *
            FROM {raw_source}
            WHERE list_contains("countries_tags", 'en:philippines')
        ) TO '{PH_RAW_PARQUET_PATH.as_posix()}'
        (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 10000)
        """
    )

## Clean up raw data

uncomment the code to remove the large dataset

In [ ]:
from src.config import RAW_PARQUET_PATH

# RAW_PARQUET_PATH.unlink(missing_ok=True)